# Вспомогательный код

Чтобы результаты экспериментов воспроизводились, зафиксируем seed's:

In [1]:
import torch
import random
import numpy as np


def set_random_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


set_random_seed(42)

Для выполнения задания рекомендуется использовать среду с аппаратным ускорителем GPU:

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


Установим Transformers и необходимые библиотеки:

In [3]:
from IPython.display import clear_output

!pip install -q -U transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q datasets evaluate

clear_output()

In [4]:
!pip install -q trl
!pip install -q bitsandbytes einops #wndb
!pip install -q peft

# Задание 1. Генерация текста

Возьмите произведение Гете "Фауст" и обучите на нем модель генерации текста.

При обучении игнорируйте знаки препинания и номера страниц.

Используйте предобученную модель `sberbank-ai/rugpt3small_based_on_gpt2` из библиотеки Hugging Face. Попробуйте разные параметры модели (например, разные режимы генерации). Как меняется результат?


Импорт необходимых библиотек:

In [5]:
import torch
from transformers import Trainer, TrainingArguments
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import Dataset
from IPython.display import clear_output

## Загрузка и подготовка данных

In [15]:
!wget -q https://edunet.kea.su/repo/EduNet-web_dependencies/datasets/Faust.txt

In [16]:
with open("Faust.txt") as text_file:
    faust_text = "".join(text_file.readlines())

In [17]:
faust_text

'вы вновь со мной, туманные виденья,\nмне в юности мелькнувшие давно…\nвас удержу ль во власти вдохновенья?\nбылым ли снам явиться вновь дано?\nиз сумрака, из тьмы полузабвенья\nвосстали вы… о, будь, что суждено!\nкак в юности, ваш вид мне грудь волнует,\nи дух мой снова чары ваши чует.\nвы принесли с собой воспоминанье\nвеселых дней и милых теней рой;\nвоскресло вновь забытое сказанье\nлюбви и дружбы первой предо мной;\nвсе вспомнилось: и прежнее страданье,\nи жизни бег запутанной чредой,\nи образы друзей, из жизни юной\nисторгнутых, обманутых фортуной.\nкому я пел когда-то, вдохновенный,\nтем песнь моя – увы! – уж не слышна…\nкружок друзей рассеян по вселенной,\nих отклик смолк, прошли те времена.\nя чужд толпе со скорбью, мне священной,\nмне самая хвала ее страшна,\nа те, кому моя звучала лира,\nкто жив еще, – рассеяны средь мира.\nи вот воскресло давнее стремленье\nтуда, в мир духов, строгий и немой,\nи робкое родится песнопенье,\nстеня, дрожа эоловой струной;\nв суровом сердце тре

In [18]:
len(faust_text.split(" "))

51923

Удалим знаки управляющих символов и приведем текст к нижнему регистру:

In [19]:
text = (
    faust_text.replace("\n", " ")
    .replace("\r", "")
    .replace("\ufeff", "")
    .replace("\x0c", "")
)
text = "".join([char for char in text if char.isalpha() or char == " "])
text = text.lower()

In [11]:
text

'вы вновь со мной туманные виденья мне в юности мелькнувшие давно вас удержу ль во власти вдохновенья былым ли снам явиться вновь дано из сумрака из тьмы полузабвенья восстали вы о будь что суждено как в юности ваш вид мне грудь волнует и дух мой снова чары ваши чует вы принесли с собой воспоминанье веселых дней и милых теней рой воскресло вновь забытое сказанье любви и дружбы первой предо мной все вспомнилось и прежнее страданье и жизни бег запутанной чредой и образы друзей из жизни юной исторгнутых обманутых фортуной кому я пел когдато вдохновенный тем песнь моя  увы  уж не слышна кружок друзей рассеян по вселенной их отклик смолк прошли те времена я чужд толпе со скорбью мне священной мне самая хвала ее страшна а те кому моя звучала лира кто жив еще  рассеяны средь мира и вот воскресло давнее стремленье туда в мир духов строгий и немой и робкое родится песнопенье стеня дрожа эоловой струной в суровом сердце трепет и смиренье в очах слеза сменяется слезой все чем владею вдаль кудато 

Запишем обработанные данные в текстовый файл:

In [20]:
train_path = "faust_train.txt"
with open(train_path, "w") as f:
    f.write(text)

torch.Size([1, 45])

## Предобученная sberbank-ai GPT2

Возьмите предобученную модель и попробуйте сгенерировать текст с ее помощью. Оцените результаты.

Весь текст подать на вход не получится. Подайте первые 30 слов.

In [29]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel


model_name_or_path = "sberbank-ai/rugpt3small_based_on_gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name_or_path)
model = GPT2LMHeadModel.from_pretrained(model_name_or_path)

first_30_words = " ".join(text.split()[:30])
input_ids = tokenizer.encode(first_30_words, return_tensors="pt")

sample_output = model.generate(
    input_ids,
    do_sample=False,
    max_length=100,
    num_beams=1,
    pad_token_id=tokenizer.eos_token_id,
)
generated_text = tokenizer.decode(sample_output[0], skip_special_tokens=True)
clear_output()

In [30]:
print(generated_text)

вы вновь со мной туманные виденья мне в юности мелькнувшие давно вас удержу ль во власти вдохновенья былым ли снам явиться вновь дано из сумрака из тьмы полузабвенья восстали вы в вышних, и в вышних, и в вышних, и в вышних, и в вышних, и в вышних, и в вышних, и в вышних, и в вышних, и в вышних, и в вышних, и


## Файнтюнинг sberbank-ai GPT2

Дообучите модель. Для этого сначала создадим датасет и `data_collator`, который нарезает текст на оптимальные по длине куски:

In [32]:
from transformers import  DataCollatorForLanguageModeling

class TextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=64):
        with open(file_path, encoding="utf-8") as f:
            text = f.read()
        tokenized = tokenizer(text, return_tensors="pt")["input_ids"][0]
        self.examples = [
            tokenized[i : i + block_size]
            for i in range(0, len(tokenized) - block_size + 1, block_size)
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return {"input_ids": self.examples[i], "labels": self.examples[i].clone()}


train_path = "Faust.txt"


train_dataset = TextDataset(tokenizer=tokenizer, file_path=train_path, block_size=64)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
clear_output()

In [37]:
train_dataset.__len__()

1771

In [48]:
from transformers import Trainer, TrainingArguments
from transformers import get_linear_schedule_with_warmup
model.to(device)

# Your code here
training_args = TrainingArguments(
    output_dir='./res_task_1_fine_tune_sberbank-ai GPT2',
    #overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=1, # number of training epochs
    per_device_train_batch_size=8, # batch size for training
    per_device_eval_batch_size=8,  # batch size for evaluation
    warmup_steps=10,# number of warmup steps for learning rate scheduler
    gradient_accumulation_steps=4, # to make "virtual" batch size larger
    report_to="none",
)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3)
total_steps = train_dataset.__len__() // 32
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=training_args.warmup_steps,
    num_training_steps=total_steps,
)


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    optimizers = [optimizer, scheduler] # Your code here # Optimizer and lr scheduler
)

In [49]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=56, training_loss=4.294544492449079, metrics={'train_runtime': 41.1047, 'train_samples_per_second': 43.085, 'train_steps_per_second': 1.362, 'total_flos': 84095865520128.0, 'train_loss': 4.294544492449079, 'epoch': 1.0})

In [50]:
# Probability beam sampling example
text = "Ходил"
input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    out = model.generate(
        input_ids,
        do_sample=True,
        num_beams=2,
        temperature=1.5,
        top_p=0.9,
        max_length=200,
    )

generated_text = list(map(tokenizer.decode, out))[0]

In [51]:
print(generated_text)

Ходил бы ты, милый!
да, я не прочь, чтоб ты был!
тебе бы в ту ночь было,
как я вижу, лучше, чем мне.
а теперь, увы!
мне так жаль!
с тех пор, как был в нем,
он стал мне сниться,
как наяву.
я знаю, где он,
где ты;
он там; я его знаю,
а он нет.
я знаю, кто он,
кто он.
он мне знаком; я слышал его,
но, к счастью, он мне не знаком,
я не слышал его и не знаю.
ну, я пойду, милый,
посмотрим, где он прячется!
и, клянусь, ты знаешь, где он!
как ты его знаешь?
что ж, он здесь! я его знаю.
я знаю, где он; я его знаю,
а он нигде не живет;
я слышал


## Другая модель с HuggingFace

In [52]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel


tokenizer = GPT2Tokenizer.from_pretrained(
    "ai-forever/rugpt3small_based_on_gpt2", clean_up_tokenization_spaces=False
)
model = GPT2LMHeadModel.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
from transformers import DataCollatorForLanguageModeling

train_path = "Faust.txt"

train_dataset = TextDataset(tokenizer=tokenizer, file_path=train_path, block_size=64)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
clear_output()

In [55]:
from transformers import Trainer, TrainingArguments
from transformers import get_linear_schedule_with_warmup
model.to(device)

# Your code here
training_args = TrainingArguments(
    output_dir='./res_task_1_fine_tune_rugpt3small_based_on_gpt2',
    #overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=1, # number of training epochs
    per_device_train_batch_size=8, # batch size for training
    per_device_eval_batch_size=8,  # batch size for evaluation
    warmup_steps=10,# number of warmup steps for learning rate scheduler
    gradient_accumulation_steps=4, # to make "virtual" batch size larger
    report_to="none",
)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
total_steps = train_dataset.__len__() // 32
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=training_args.warmup_steps,
    num_training_steps=total_steps,
)


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    optimizers = [optimizer, scheduler] # Your code here # Optimizer and lr scheduler
)

In [56]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=56, training_loss=4.294544492449079, metrics={'train_runtime': 47.0256, 'train_samples_per_second': 37.66, 'train_steps_per_second': 1.191, 'total_flos': 84095865520128.0, 'train_loss': 4.294544492449079, 'epoch': 1.0})

In [57]:
# Probability beam sampling example
text = "все"
input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    out = model.generate(
        input_ids,
        do_sample=True,
        num_beams=1,
        temperature=1.6,
        top_p=0.7,
        max_length=100,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = list(map(tokenizer.decode, out))[0]

In [58]:
print(generated_text)

все
так же сильно ты был прекрасен, и
все, что для тебя сейчас, не так уж плохо!
мне было с тобою хорошо:
и тебе, и ему я рад,
что мы еще так прекрасны.
но, когда ты был так слаб,
как это случилось, мы еще так сильны.
ты должен был нам все дать,
так ты нам все испортил;
у нас нет и сил, нет и любви.
нет, ничего не выйдет


In [59]:
del model
del tokenizer

torch.cuda.empty_cache()

## Формат результата

Сгенерированный текст

Пример текста:

"все все от бесстыдные старой

все в нем получше все стремленья

поддержки с собой в сердце воздух своей

и в вечной страсти восстанет свой предлог

привет вам слуга в сладком страшней стране

и в мире все вражда станет станет

в поле на пользу своим воспоминанья"


# Задание 2. Классификация с помощью BERT

Возьмите набор данных эмоциональных окрасок отзывов [emotions dataset 🛠️[doc]](https://huggingface.co/datasets/emotion). В датасете 5 классов. Получите эмбеддинг из BERT-подобного кодировщика. Классифицируйте тексты с помощью методов ML или  нейросети, используя эмбеддинги в качестве входов.

Произведите fine-tuning кодировщика на ваших данных (классификатор на основе BERT) и сравните, как изменилось качество классификации.

Импорт необходимых библиотек:

In [3]:
import torch
import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset
from datasets import load_dataset
from datasets.dataset_dict import DatasetDict
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModel
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from IPython.display import clear_output

Взглянем на датасет.

In [2]:
train = load_dataset("SetFit/emotion", split="train")
clear_output()

train_df = pd.DataFrame({"text": train["text"], "labels": train["label"]})
train_df.head()

,text,labels
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


Инициализация токенайзера и модели. Перед тем, как отправить текст в модель, его следует токенизировать.

**Учитывайте**, что:
* Мы посмотрели несколько способов получения эмбеддингов из моделей BERT. Воспользуйтесь одним из них.
* Длина эмбеддинга больше 200 в данной задаче не потребуется. Чем длиннее эмбеддинг, тем медленнее всё будет учиться.
* В каждой части задания нужно задать по отдельной модели.
* Библиотеки в первой и второй части задания могут конфликтовать. Используйте "перезапуск сеанса", если так случилось.

Возьмите модель `bert-tiny` и соответствующий токенизатор.

In [7]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
model = AutoModel.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
clear_output()
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-1): 2 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=128, out_features=128, bias=True)
            (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

Посмотрим, как устроен датасет и какой у него баланс классов:

In [10]:
# Your code here
print("Размер датасета:", len(train_df))
print("Баланс:")
print(train_df["labels"].value_counts().sort_index())

Размер датасета: 16000
Баланс:
labels
0    4666
1    5362
2    1304
3    2159
4    1937
5     572
Name: count, dtype: int64


Оставим 5000 объектов, поделив их на обучение, валидацию, тест.

Будем семплировать, в функцию `sample` из `pandas` подадим веса для каждого объекта. Создадим столбец, в каждой строке которого будет частота семплирования.

In [11]:
train_df["freq"] = len(train_df) / train_df.groupby("labels")["labels"].transform("count")
sampledf = train_df.sample(weights=train_df.freq)

In [12]:
sampledf = train_df.sample(5000, weights=train_df.freq)
sampledf["labels"].value_counts()

,count
labels,
0,1010
1,991
3,884
4,837
2,780
5,498


Также имеет смысл проверить, что в колонке `labels` данные типа `int`. Если нет, то привести к этому типу.

In [13]:
sampledf["labels"] = sampledf["labels"].astype(int)

sample = list(sampledf["text"])
labels = list(sampledf["labels"])

Разделите данные на `train`, `val`, `test`. Получите `x_train`, `y_train` и т.д.

In [14]:
from sklearn.model_selection import train_test_split

# Your code here
x_train, x_temp, y_train, y_temp = train_test_split(
    sample, labels, test_size=0.2, random_state=42, stratify=labels
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(x_train)}, Val: {len(x_val)}, Test: {len(x_test)}")


Train: 4000, Val: 500, Test: 500


Проверьте, какие объекты и в каком количестве представлены в метках в трёх выборках.

In [16]:
# Your code here
print("Train")
print(pd.Series(y_train).value_counts().sort_index())
print("Val")
print(pd.Series(y_val).value_counts().sort_index())
print("Tes")
print(pd.Series(y_test).value_counts().sort_index())

Train
0    808
1    793
2    624
3    707
4    670
5    398
Name: count, dtype: int64
Val
0    101
1     99
2     78
3     88
4     84
5     50
Name: count, dtype: int64
Tes
0    101
1     99
2     78
3     89
4     83
5     50
Name: count, dtype: int64


Получите и сохраните векторные представления текстов, которые возвращает модель. Например, вот таким образом. В `x_train` сейчас текст в виде букв.

In [18]:
train_emb = []
for t in x_train:
    encoded = tokenizer.encode(t, return_tensors="pt").to(device)
    output = model(encoded)["pooler_output"][0].detach().cpu()
    train_emb.append(output)

train_emb = torch.stack(train_emb).numpy()

In [19]:
val_emb = []
for t in x_val:
    encoded = tokenizer.encode(t, return_tensors="pt").to(device)
    output = model(encoded)["pooler_output"][0].detach().cpu()
    val_emb.append(output)

val_emb = torch.stack(val_emb).numpy()

In [20]:
test_emb = []
for t in x_test:
    encoded = tokenizer.encode(t, return_tensors="pt").to(device)
    output = model(encoded)["pooler_output"][0].detach().cpu()
    test_emb.append(output)

test_emb = torch.stack(test_emb).numpy()

Возьмите ML-классификатор, обучите его и постройте `classification_report`.

Базово можно взять `RandomForestClassifier`.

In [21]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(train_emb, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [22]:
from sklearn.metrics import classification_report

y_pred = clf.predict(test_emb)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.36      0.49      0.41       101
           1       0.39      0.46      0.42        99
           2       0.45      0.37      0.41        78
           3       0.26      0.28      0.27        89
           4       0.31      0.28      0.29        83
           5       0.70      0.14      0.23        50

    accuracy                           0.36       500
   macro avg       0.41      0.34      0.34       500
weighted avg       0.39      0.36      0.35       500



Здесь не требуется получить 100% Presicion и Recall. Однако проверьте, что вы верно разделили данные на 3 подвыборки и что значения метрик не находятся в районе нуля.

**Попробуйте дообучить BERT и произвести классификацию заново.**

Загрузим данные заново. В этом разделе необходимо создать `Dataset` в формате `HuggingFace`.

In [6]:
train = load_dataset("SetFit/emotion", split="train")
clear_output()

train_df = pd.DataFrame({"text": train["text"], "labels": train["label"]})
train_df.head()

,text,labels
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


Заново разделите данные на данные на `train`, `val`, `test`, как делали выше. Мы вновь начинаем с текстов в привычном буквенном формате.

In [7]:
# Your code here
train_df["freq"] = len(train_df) / train_df.groupby("labels")["labels"].transform("count")
sampledf = train_df.sample(5000, weights=train_df.freq, random_state=42)
sampledf["labels"] = sampledf["labels"].astype(int)

sample = list(sampledf["text"])
labels = list(sampledf["labels"])

x_train, x_temp, y_train, y_temp = train_test_split(
    sample, labels, test_size=0.2, random_state=42, stratify=labels
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)



In [8]:
import evaluate

metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(
        predictions=predictions, references=labels, average="weighted"
    )

In [9]:
from datasets import Dataset
from datasets.dataset_dict import DatasetDict

d = {
    "train": Dataset.from_dict({"label": y_train, "text": x_train}),
    "val": Dataset.from_dict({"label": y_val, "text": x_val}),
    "test": Dataset.from_dict({"label": y_test, "text": x_test}),
}

dataset = DatasetDict(d)

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 4000
    })
    val: Dataset({
        features: ['label', 'text'],
        num_rows: 500
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 500
    })
})

Настройте токенизатор (если это не было сделано выше) так, чтобы производились дополнение (padding) и обрезка предложений. Имеет смысл указать длину получаемых векторных представлений (`model_max_length`).

In [11]:
# Your code here
tokenizer_bert = AutoTokenizer.from_pretrained(
    "google/bert_uncased_L-2_H-128_A-2",
    model_max_length=128,
)

Напишите функцию, которая будет токенизировать ваш текст в `DatasetDict`.

In [12]:
# Your code here
def tokenize_function(examples):
    return tokenizer_bert(
        examples["text"],
        padding="max_length",
        truncation=True,
    )

Токенизируйте ваши данные:

In [13]:
# Your code here
tokenized_data = dataset.map(tokenize_function, batched=True)
tokenized_data = tokenized_data.rename_column("label", "labels")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Посмотрите, как  выглядит ваш `DatasetDict`, какие размеры (`shape`) сущностей, которые он содержит. Выведите первый объект из `train`.

Там должен находиться как исходный текст, так и его представление в виде токенов (из токенизатора, не из модели).

In [14]:
# Your code here
print("struct:")
print(tokenized_data)

struct:
DatasetDict({
    train: Dataset({
        features: ['labels', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4000
    })
    val: Dataset({
        features: ['labels', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
    test: Dataset({
        features: ['labels', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
})


Импортируйте модель с учётом того, что вы уже решаете не просто задачу построения эмбеддингов, а задачу классификации. Укажите верное число классов в ней.

In [23]:
from transformers import AutoModelForSequenceClassification

model_for_classification = AutoModelForSequenceClassification.from_pretrained(
    "google/bert_uncased_L-2_H-128_A-2",
    num_labels=6,
)
model_for_classification.to(device)

for param in model_for_classification.parameters():
    param.data = param.data.contiguous()

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
# model

Задайте параметры, обучите модель:

In [28]:
from transformers import Trainer, TrainingArguments

# Your code here
training_args = TrainingArguments(
    output_dir="./my_awesome_model",
    learning_rate=2e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3, #,#about 1.5 minutes for 1 epoch
    weight_decay=0.01,#,
    eval_strategy="epoch",#,
    save_strategy="epoch",#,
    load_best_model_at_end=True,#,
    push_to_hub=False,
    report_to="none",
)



trainer = Trainer(
    model=model_for_classification,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=tokenized_data['train'],         # training dataset
    eval_dataset=tokenized_data['val'],
    processing_class=tokenizer_bert,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,No log,0.670676,0.800336
2,No log,0.483667,0.860474
3,No log,0.459762,0.879648


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.en

TrainOutput(global_step=375, training_loss=0.6778890787760417, metrics={'train_runtime': 13.4207, 'train_samples_per_second': 894.142, 'train_steps_per_second': 27.942, 'total_flos': 3816216576000.0, 'train_loss': 0.6778890787760417, 'epoch': 3.0})

Мы дообучили модель! Теперь проверим качество предсказания на тесте.

Можете использовать `trainer.predict`.

In [29]:
# Your code here
predictions = trainer.predict(tokenized_data["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

In [30]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.86      0.91      0.89        98
           1       0.94      0.83      0.88        99
           2       0.85      0.97      0.91        77
           3       0.91      0.90      0.90        88
           4       0.87      0.76      0.81        88
           5       0.84      0.98      0.91        50

    accuracy                           0.88       500
   macro avg       0.88      0.89      0.88       500
weighted avg       0.88      0.88      0.88       500



Если у вас получилось 100%, проверьте, что с чем вы сравниваете. По умолчанию `predict` возвращает логиты и истинные метки, которые были изначально в переданном объекте.

## Формат результата

Получить значение качества классификации.